##### Import statements:

In [ ]:
import os
import pathlib as Path
import sys
import socket
import time
import numpy as np
import pandas as pd
from data_analysis_tools_mkTurk.utils_meta import get_recording_path
from data_analysis_tools_mkTurk.npix import read_recording_coordinate_data_sheet    
from data_analysis_tools_mkTurk.IO import ch_dicts_2_h5

##### Define inputs, parameters:

In [ ]:
# Define sessions:
#sessions = "Expt_num=='E7' and monkey=='Bourgeois' and series_sess_num==1 and date >= '20250904' and date=='20251125'" 
#sessions = "Expt_num=='E6' and monkey=='Bourgeois' and date=='20250818'" 
#sessions = ""Expt_num=='E7' and series_sess_num==0 and date < '20250904' and date not in ['20250103', '20250107', '20250310']"
sessions = "date in ['20240308']"

# Define input paths:
base_data_path = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'Data')
preprocessed_base_path = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Younah', 'ephys')
#preprocessed_base_path = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'ephys')
chs_base_path = None
#chs_base_path = os.path.join('/', 'mnt', 'smb', 'users', 'Younah', 'ephys')


# General params:
max_ch = 384
chunk_size = 100
dtype=int
fail_on_error = True

# Output params:
save_output = True
output_base = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'ephys')

___

##### Find specific sessions to analyze:

In [ ]:
if type(sessions) == str:
    rcd = read_recording_coordinate_data_sheet()
    sessions_df = rcd.query(sessions)[['monkey', 'date']].drop_duplicates()
elif type(sessions) == list:
    sessions_df = pd.concat([pd.DataFrame(pd.Series(x)).T for x in sessions], axis=0)
sessions_df.index = np.arange(sessions_df.shape[0])

##### Hardcode some stuff:

In [ ]:
folder_level_offset = 4

##### Define main function:

In [ ]:
def main(session, base_data_path, preprocessed_base_path, channels, chunk_size, dtype, save_output, folder_level_offset=4):

    monkey = session['monkey']
    date = session['date']

    print('Saving HDF5 for session {}...'.format(date))
    recording_path = get_recording_path(Path.Path(base_data_path), Path.Path(monkey), date, depth=4)[0]
    preprocessed_data_path = os.path.join(preprocessed_base_path, monkey, recording_path.split(os.path.sep)[folder_level_offset+3])
    if chs_base_path is None:
        ch_psths_path = preprocessed_data_path
    else:
        ch_psths_path = os.path.join(preprocessed_base_path, monkey, recording_path.split(os.path.sep)[folder_level_offset+3])

    output_directory = os.path.join(output_base, monkey, recording_path.split(os.path.sep)[folder_level_offset+3])
    
    start = time.time()
    trial_info, spikes_counts = ch_dicts_2_h5(base_data_path, monkey, date, preprocessed_data_path=preprocessed_data_path, channels=channels, chunk_size=chunk_size, dtype=dtype, save_output=save_output, fname=date, output_directory=output_directory)
    stop = time.time()


##### Iterate over sessions:

In [ ]:
channels = np.arange(max_ch)

if fail_on_error:
    for s, session in sessions_df.iterrows():
        main(session, base_data_path, preprocessed_base_path, channels, chunk_size, dtype, save_output, folder_level_offset)

else:
    errors = []
    for s, session in sessions_df.iterrows():
        try:
            main(session, base_data_path, preprocessed_base_path, channels, chunk_size, dtype, save_output, folder_level_offset)
    
        except Exception as e:
            errors.append(('{}, {}'.format(session.monkey, session.date), e))
            continue
        
    # Print exceptions:
    if len(errors) > 0:
        print('Encountered following exceptions:')
        for err in errors:
            print('{} : {}\n'.format(err[0], err[1]))
        
    """    
    #try:
    monkey = session['monkey']
    date = session['date']

    print('Saving HDF5 for session {}...'.format(date))
    recording_path = get_recording_path(Path.Path(base_data_path), Path.Path(monkey), date, depth=4)[0]
    #preprocessed_data_path = os.path.join('C:\\', 'Users', 'danie', 'Documents','test_read_dicts_local')
    preprocessed_data_path = os.path.join(preprocessed_base_path, monkey, recording_path.split(os.path.sep)[folder_level_offset+3])
    if chs_base_path is None:
        ch_psths_path = preprocessed_data_path
    else:
        ch_psths_path = os.path.join(preprocessed_base_path, monkey, recording_path.split(os.path.sep)[folder_level_offset+3])

    output_directory = os.path.join(output_base, monkey, recording_path.split(os.path.sep)[folder_level_offset+3])
    start = time.time()
    trial_info, spikes_counts = ch_dicts_2_h5(base_data_path, monkey, date, preprocessed_data_path=preprocessed_data_path, channels=channels, chunk_size=chunk_size, dtype=dtype, save_output=save_output, fname=date, output_directory=output_directory)
    stop = time.time()
    # Capture any exception:
    #except Exception as e:
    #    errors.append(('{}, {}'.format(monkey, date), e))
    #    continue
    print('Write duration = {} minutes'.format( (stop-start)/60 ))
    """